# ST456 极简一键运行 Colab Notebook

Colab 提示上传文件时，上传项目 ZIP，然后选择 `Runtime > Run all`。

In [ ]:
from pathlib import Path
import json
import os
import shutil
import subprocess
import sys
import zipfile

# 显示进度条
os.environ['HF_HUB_DISABLE_PROGRESS_BARS'] = '1'
os.environ['PYTHONUNBUFFERED'] = '1'

# 打印命令，方便定位
def run(cmd):
    print(f'\n>>> {cmd}', flush=True)
    subprocess.run(cmd, shell=True, check=True)

# 清理gpu缓存。
def clear_gpu_memory(stage):
    import gc

    gc.collect()
    try:
        import torch
    except ImportError:
        return
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        if hasattr(torch.cuda, 'ipc_collect'):
            torch.cuda.ipc_collect()
        print(f'[GPU cleanup] {stage}')


## 上传并配置环境

In [ ]:
from google.colab import files

workspace_root = Path('/content')
os.chdir(workspace_root)

# 上传最终提交用的项目 ZIP。
uploaded = files.upload()
zip_files = [name for name in uploaded if name.lower().endswith('.zip')]
if not zip_files:
    raise ValueError('Please upload the project ZIP file.')

# 直接搜索text/。
with zipfile.ZipFile(zip_files[0], 'r') as archive:
    archive.extractall(workspace_root)

project_dirs = sorted(path for path in workspace_root.rglob('text') if path.is_dir())
if not project_dirs:
    raise FileNotFoundError('Could not find the text directory after extraction.')

project_root = project_dirs[0]
os.chdir(project_root)
# 把 src/ 加进路径。
sys.path.insert(0, str(project_root / 'src'))

print('Project directory:', project_root)
run(f'{sys.executable} -m pip install -r requirements.txt')
# 删掉 Colab 旧版 torchao；这里的 PEFT LoRA 不需要它。
run(f'{sys.executable} -m pip uninstall -y torchao')


## 构建数据

In [ ]:
# 用 Gutenberg 文本重新构建数据。
run(f'{sys.executable} scripts/download_gutenberg.py')
run(f'{sys.executable} scripts/build_dataset.py --context-size 4 --min-chars 40')

# 这些 token 检查对应报告里的 prompt 长度 sanity check。
Path('artifacts/eval').mkdir(parents=True, exist_ok=True)
for experiment_id, config_path in [
    ('E1', 'configs/e1_distilgpt2_plain_full.yaml'),
    ('E3', 'configs/e3_distilgpt2_structured_long_context.yaml'),
    ('E5', 'configs/e5_distilgpt2_structured_aux_ranking.yaml'),
]:
    output_path = Path('artifacts/eval') / f'token_stats_{experiment_id.lower()}.json'
    run(f'{sys.executable} scripts/inspect_token_stats.py --config {config_path} --output-path {output_path}')


## 训练和评估

In [ ]:
main_experiments = [
    ('E1', 'configs/e1_distilgpt2_plain_full.yaml', 'artifacts/e1_plain_full'),
    ('E2', 'configs/e2_distilgpt2_structured_full.yaml', 'artifacts/e2_structured_full'),
    ('E3', 'configs/e3_distilgpt2_structured_long_context.yaml', 'artifacts/e3_long_context'),
    ('E4', 'configs/e4_distilgpt2_structured_lora.yaml', 'artifacts/e4_lora'),
    ('E5', 'configs/e5_distilgpt2_structured_aux_ranking.yaml', 'artifacts/e5_aux_ranking'),
    ('E5W', 'configs/e5_distilgpt2_structured_aux_ranking_wide.yaml', 'artifacts/e5_aux_ranking_wide'),
]

# 每个模型按自己的 YAML 配置训练，跑完就清一次 GPU 。
for experiment_id, config_path, _model_dir in main_experiments:
    print(f'----- {experiment_id}: training -----')
    run(f'{sys.executable} scripts/train_experiment.py --config {config_path} --seed 42')
    clear_gpu_memory(f'{experiment_id} training finished')

eval_output_dir = Path('artifacts/eval')
eval_output_dir.mkdir(parents=True, exist_ok=True)

# 用 3 个 generation seed 做评估。
for experiment_id, _config_path, model_dir in main_experiments:
    print(f'----- {experiment_id}: 3-seed evaluation -----')
    run(
        f'{sys.executable} scripts/run_eval_3seed.py '
        f'--experiment-id {experiment_id.lower()} '
        f'--model-dir {model_dir} '
        f'--output-dir {eval_output_dir}'
    )
    clear_gpu_memory(f'{experiment_id} evaluation finished')


## Retrieval 附录实验

In [ ]:
# Retrieval 单独跑，因为它是附录对比，不是主线最佳模型。
run(f'{sys.executable} scripts/train_retrieval_model.py --config configs/retrieval_distilgpt2.yaml')
run(f'{sys.executable} scripts/generate_samples.py --model-dir artifacts/retrieval --use-retrieval --history-path data/processed/train.jsonl --output-path artifacts/eval/generated_samples_retrieval.jsonl')
run(f'{sys.executable} scripts/run_auto_eval.py --model-dir artifacts/retrieval --generated-path artifacts/eval/generated_samples_retrieval.jsonl --output-path artifacts/eval/metrics_retrieval.csv')
clear_gpu_memory('retrieval appendix finished')


## 下载结果

In [ ]:
# 删掉 checkpoint。
for checkpoint_dir in sorted(Path('artifacts').rglob('checkpoint-*')):
    if checkpoint_dir.is_dir():
        shutil.rmtree(checkpoint_dir)

download_dir = Path('artifacts/download_package')
download_dir.mkdir(parents=True, exist_ok=True)

# 保留：分数、生成样本、token 统计和训练配置。
eval_src = Path('artifacts/eval')
if eval_src.exists():
    shutil.copytree(eval_src, download_dir / 'eval', dirs_exist_ok=True)

for config_file in Path('artifacts').glob('*/training_config.json'):
    dest = download_dir / config_file.parent.name / config_file.name
    dest.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(config_file, dest)

# 下载一个 ZIP。
archive_path = shutil.make_archive(str(Path('artifacts') / 'colab_results'), 'zip', root_dir=str(download_dir))
shutil.rmtree(download_dir)

print('Results ZIP:', archive_path)
files.download(archive_path)
